In [0]:
%sql
SELECT current_catalog() AS current_catalog

current_catalog
workspace


In [0]:

# Identify your workspace catalog
catalog = spark.sql("SELECT current_catalog()").first()[0]

# Create a dedicated schema for our hospital project
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.hospital_project")

# Create a volume for input CSV files
spark.sql(
    f"CREATE VOLUME IF NOT EXISTS "
    f"`{catalog}`.hospital_project.source_files"
)

print("Catalog:", catalog)
print("Hospital project schema and volume created!")


Catalog: workspace
Hospital project schema and volume created!


In [0]:

# Path to patient CSV in Unity Catalog Volume
patient_path = (
    f"/Volumes/{catalog}/hospital_project/"
    f"source_files/patients.csv"
)

# Read the CSV
patients_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(patient_path)
)

# Display the records
display(patients_df)

# Check the structure
patients_df.printSchema()


patient_id,patient_name,age,gender,city,registration_date
P001,Demo Patient 001,32,Male,Chennai,2026-03-04
P002,Demo Patient 002,46,Male,Bengaluru,2026-05-20
P003,Demo Patient 003,29,Other,Mysuru,2026-01-09
P004,Demo Patient 004,21,Male,Hyderabad,2026-03-01
P005,Demo Patient 005,82,Other,Bengaluru,2026-05-24
P006,Demo Patient 006,43,Other,Mysuru,2026-02-26
P007,Demo Patient 007,75,Other,Chennai,2026-01-02
P008,Demo Patient 008,38,Other,Mysuru,2026-03-29
P009,Demo Patient 009,53,Male,Hyderabad,2026-03-28
P010,Demo Patient 010,31,Male,Mysuru,2026-01-25


root
 |-- patient_id: string (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- city: string (nullable = true)
 |-- registration_date: date (nullable = true)



In [0]:

catalog = spark.sql("SELECT current_catalog()").first()[0]
schema = "hospital_project"

source_path = f"/Volumes/{catalog}/{schema}/source_files"

print("Catalog:", catalog)
print("Schema:", schema)
print("Source:", source_path)


Catalog: workspace
Schema: hospital_project
Source: /Volumes/workspace/hospital_project/source_files


In [0]:

patients_df = (
    spark.read
    .option("header", "true")
    .csv(f"{source_path}/patients.csv")
)

appointments_df = (
    spark.read
    .option("header", "true")
    .csv(f"{source_path}/appointments.csv")
)

treatments_df = (
    spark.read
    .option("header", "true")
    .csv(f"{source_path}/treatments.csv")
)

print("Patients:", patients_df.count())
print("Appointments:", appointments_df.count())
print("Treatments:", treatments_df.count())


Patients: 53
Appointments: 92
Treatments: 44


In [0]:

patients_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"`{catalog}`.{schema}.bronze_patients")

appointments_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"`{catalog}`.{schema}.bronze_appointments")

treatments_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"`{catalog}`.{schema}.bronze_treatments")

print("All three Bronze Delta tables created successfully!")


All three Bronze Delta tables created successfully!


In [0]:

display(
    spark.table(f"`{catalog}`.{schema}.bronze_patients")
)

display(
    spark.table(f"`{catalog}`.{schema}.bronze_appointments")
)

display(
    spark.table(f"`{catalog}`.{schema}.bronze_treatments")
)


patient_id,patient_name,age,gender,city,registration_date
P001,Demo Patient 001,32,Male,Chennai,2026-03-04
P002,Demo Patient 002,46,Male,Bengaluru,2026-05-20
P003,Demo Patient 003,29,Other,Mysuru,2026-01-09
P004,Demo Patient 004,21,Male,Hyderabad,2026-03-01
P005,Demo Patient 005,82,Other,Bengaluru,2026-05-24
P006,Demo Patient 006,43,Other,Mysuru,2026-02-26
P007,Demo Patient 007,75,Other,Chennai,2026-01-02
P008,Demo Patient 008,38,Other,Mysuru,2026-03-29
P009,Demo Patient 009,53,Male,Hyderabad,2026-03-28
P010,Demo Patient 010,31,Male,Mysuru,2026-01-25


appointment_id,patient_id,department,appointment_date,status
A001,P005,Cardiology,2026-07-13,Completed
A002,P033,Orthopedics,2026-07-06,Cancelled
A003,P014,Dermatology,2026-06-17,Cancelled
A004,P016,General Medicine,2026-07-23,Completed
A005,P007,Cardiology,2026-07-26,Scheduled
A006,P028,General Medicine,2026-07-30,Completed
A007,P044,Cardiology,2026-06-08,Cancelled
A008,P047,Neurology,2026-06-14,Completed
A009,P013,Orthopedics,2026-08-08,Cancelled
A010,P009,General Medicine,2026-06-24,Scheduled


treatment_id,appointment_id,treatment_type,cost
T001,A001,Blood Test,1200
T002,A004,ECG,1200
T003,A006,Physiotherapy,500
T004,A008,Blood Test,750
T005,A012,Blood Test,1800
T006,A015,Blood Test,3200
T007,A017,Blood Test,500
T008,A018,X-Ray,1800
T009,A021,ECG,2500
T010,A022,X-Ray,1800


In [0]:

tables = [
    "bronze_patients",
    "bronze_appointments",
    "bronze_treatments"
]

for table in tables:
    df = spark.table(f"`{catalog}`.{schema}.{table}")
    print(f"{table}: {df.count()} records")


bronze_patients: 53 records
bronze_appointments: 92 records
bronze_treatments: 44 records


In [0]:

display(
    spark.sql(
        f"DESCRIBE DETAIL `{catalog}`.{schema}.bronze_patients"
    )
)


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,c9af4219-b9c0-4dd8-b5d6-f57a74813d26,workspace.hospital_project.bronze_patients,null,,2026-11-02T05:50:36.112Z,2026-11-03T14:40:36.000Z,List(),List(),1,2849,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.workloadBasedColumns.deltaFileStatistics -> `age`, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> a843c6fc-f01a-43b2-8253-e77584202f10, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-65377c4a-9583-40ac-b5b7-0adee95150a7, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-f9b6ca73-43ad-4144-baa6-241239bbcd72)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",true
